# Галюцинації з високою оцінкою судді

Суддя бачить лише текст, тож граматично правильна галюцинація вчителя (наприклад «— Усе в порядку!» на тиші чи музиці) отримує 4–5. Confidence вчителя у таких слайсів низька.

Ноутбук збирає слайси з **confidence < `CONF_MAX`** і **оцінкою судді ≥ `MARK_MIN`** для ручної перевірки. Дані ті самі, що в `judge_vs_confidence.py`: найновіший `label_marks.csv`, confidence з `labels.csv` того ж вчителя, лише слайси з однаковим текстом.

In [1]:
CONF_MAX = 0.85   # confidence нижче цього
MARK_MIN = 4      # оцінка судді не нижче цього
SAMPLE = 200      # скільки випадкових рядків показати
SEED = 0

In [2]:
import sys
from pathlib import Path

import pandas as pd

HERE = Path.cwd()
if not (HERE / "judge_vs_confidence.py").is_file():
    HERE = Path("pseudo-labelling/confidences").resolve()
sys.path.insert(0, str(HERE))
from judge_vs_confidence import ROOT, confidence_labels, load, newest_marks, read_csv

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)

marks_path = newest_marks()
labels_path = confidence_labels(marks_path)
frame, info = load(marks_path, labels_path, any_text=False)
# path і timestamp, щоб знайти слайс в аудіо вручну.
places = read_csv(labels_path, ["ID", "path", "timestamp"])
frame = frame.merge(places, on="ID", how="left")
print(f"marks: {marks_path.relative_to(ROOT)}\nconfidence: {labels_path.relative_to(ROOT)}")
print(f"слайсів для порівняння: {len(frame):,}")

marks: output/pseudo-labelling/model-whisper-m-fixed-leaks-10-epochs-epoch05-01/label_marks.csv
confidence: output/pseudo-labelling/model-whisper-m-fixed-leaks-10-epochs-epoch05-02/labels.csv
слайсів для порівняння: 268,018


## Вибірка

In [3]:
selected = frame[(frame["confidence"] < CONF_MAX) & (frame["mark"] >= MARK_MIN)].copy()
low = frame[frame["confidence"] < CONF_MAX]
print(f"confidence < {CONF_MAX}: {len(low):,} слайсів, з них оцінка >= {MARK_MIN}: {len(selected):,} ({len(selected) / max(len(low), 1):.1%})")
print(f"частка від усіх слайсів: {len(selected) / len(frame):.2%}")
selected["mark"].value_counts().sort_index().rename("slices").to_frame()

confidence < 0.85: 14,335 слайсів, з них оцінка >= 4: 4,733 (33.0%)
частка від усіх слайсів: 1.77%


,slices
mark,
4,267
5,4466


## Найчастіші тексти

Однаковий текст на багатьох різних слайсах — головна ознака галюцинації.

In [4]:
repeated = (
    selected.groupby("text")
    .agg(slices=("ID", "size"), files=("path", "nunique"), mean_mark=("mark", "mean"), mean_confidence=("confidence", "mean"))
    .sort_values("slices", ascending=False)
)
print(f"унікальних текстів: {len(repeated):,}; тексти, що трапляються >= 5 разів, покривають {repeated.loc[repeated['slices'] >= 5, 'slices'].sum() / len(selected):.1%} вибірки")
repeated.head(50).style.format({"mean_mark": "{:.2f}", "mean_confidence": "{:.3f}"})

унікальних текстів: 2,043; тексти, що трапляються >= 5 разів, покривають 52.8% вибірки


,slices,files,mean_mark,mean_confidence
text,,,,
— Усе в порядку!,933,619,5.00,0.433
— Угу.,256,254,5.00,0.543
Добрий день!,95,94,5.00,0.685
— Усе в порядку.,76,71,5.00,0.433
А,70,70,4.99,0.376
Ізміст,61,61,5.00,0.541
"— Ви не знаєте, що я знаю?",60,60,5.00,0.584
— Уже йшли.,40,39,4.95,0.436
Ми,39,39,5.00,0.564


## Випадкові рядки для перевірки

Відсортовано за confidence, від найнижчої.

In [5]:
COLUMNS = ["text", "mark", "confidence", "min_logprob", "num_tokens", "compression_ratio", "path", "timestamp", "ID"]
sample = selected.sample(min(SAMPLE, len(selected)), random_state=SEED).sort_values("confidence")
sample[COLUMNS].reset_index(drop=True).style.format({"confidence": "{:.3f}", "min_logprob": "{:.2f}", "compression_ratio": "{:.2f}", "num_tokens": "{:.0f}"})

,text,mark,confidence,min_logprob,num_tokens,compression_ratio,path,timestamp,ID
0,І,5,0.262,-2.45,2,0.20,b636774b-82dc-4bb4-a8b2-5aa3c266114f.mp3,1150.000-1180.000,b636774b-82dc-4bb4-a8b2-5aa3c266114f_045
1,А,5,0.297,-2.08,1,0.20,a2e984ce-3eee-4d00-91c0-1c8b7e0a04da.mp3,0.000-0.400,a2e984ce-3eee-4d00-91c0-1c8b7e0a04da_000
2,— Він має чогось особисту.,5,0.312,-2.76,12,0.89,2b942d3a-ca05-44f4-9fc6-9d96c041b552.mp3,1730.200-1760.200,2b942d3a-ca05-44f4-9fc6-9d96c041b552_077
3,— Усе в порядку!,5,0.334,-2.77,7,0.72,2ffb0382-cf12-4e3b-ad07-42ad13e67b6b.mp3,1216.100-1227.800,2ffb0382-cf12-4e3b-ad07-42ad13e67b6b_053
4,Казані,5,0.334,-2.23,4,0.63,03370000-0aff-0242-2d36-08daf5a27006.mp3,662.900-663.700,03370000-0aff-0242-2d36-08daf5a27006_027
5,— Усе в порядку!,5,0.350,-2.10,7,0.72,4a27d8b2-b3e2-47e1-ad35-c92ca2a0644f.mp3,1775.500-1798.479,4a27d8b2-b3e2-47e1-ad35-c92ca2a0644f_076
6,— Усе в порядку!,5,0.361,-1.76,7,0.72,cb9bdd2a-b8e7-4b3f-baeb-02fa7673a396.mp3,1777.600-1798.871,cb9bdd2a-b8e7-4b3f-baeb-02fa7673a396_071
7,— Усе в порядку!,5,0.363,-2.16,7,0.72,1ad4b09e-5a34-438c-a655-1ec99aa9a0d8.mp3,1788.300-1797.879,1ad4b09e-5a34-438c-a655-1ec99aa9a0d8_076
8,Скільки,5,0.376,-1.66,3,0.61,58339f1f-16cd-40c0-958a-ead2994a144b.mp3,1683.000-1683.600,58339f1f-16cd-40c0-958a-ead2994a144b_069
9,— Убийця!..,5,0.381,-2.15,7,0.68,11d0c087-a363-4e87-8538-d00388fc4cd9.mp3,816.200-818.800,11d0c087-a363-4e87-8538-d00388fc4cd9_034


## Унікальні тексти (без найчастіших повторів)

Тексти, що трапились лише раз: серед них більше справжнього мовлення, яке вчитель розпізнав невпевнено.

In [6]:
single = selected[selected["text"].map(repeated["slices"]) == 1]
print(f"текстів, що трапились один раз: {len(single):,}")
single.sample(min(SAMPLE, len(single)), random_state=SEED).sort_values("confidence")[COLUMNS].reset_index(drop=True).style.format(
    {"confidence": "{:.3f}", "min_logprob": "{:.2f}", "compression_ratio": "{:.2f}", "num_tokens": "{:.0f}"}
)

текстів, що трапились один раз: 1,772


,text,mark,confidence,min_logprob,num_tokens,compression_ratio,path,timestamp,ID
0,Україні,5,0.342,-1.61,2,0.62,74bbac2e-07b5-45c5-97f4-06426f443ce9.mp3,217.100-247.100,74bbac2e-07b5-45c5-97f4-06426f443ce9_009
1,Уже всередині є важлива проблема!,5,0.345,-2.79,12,0.95,27275e60-5482-4b58-a19f-89ca8d57a788.mp3,0.000-21.800,27275e60-5482-4b58-a19f-89ca8d57a788_000
2,— Ми були позичені.,5,0.375,-2.24,9,0.80,7e167165-c36d-4f3c-9a2d-18015925d15b.mp3,1015.100-1043.600,7e167165-c36d-4f3c-9a2d-18015925d15b_040
3,"— Вони там, на вереві, лежать.",5,0.382,-2.43,13,0.87,61c460d3-7564-4582-8044-78452ab78d3a.mp3,30.000-60.000,61c460d3-7564-4582-8044-78452ab78d3a_001
4,— Се вже починаю.,5,0.384,-1.81,8,0.80,4480bc22-f25f-4758-898b-86e280bed99d.mp3,871.800-899.293,4480bc22-f25f-4758-898b-86e280bed99d_038
5,День півдня,5,0.391,-1.58,6,0.73,7a66d61e-8bb8-446c-bc8e-85d61bfbb692.mp3,866.500-896.500,7a66d61e-8bb8-446c-bc8e-85d61bfbb692_038
6,Горячий,5,0.401,-2.05,4,0.61,c3e42d6a-c478-4b05-9ffd-a0a05546f811.mp3,805.500-806.000,c3e42d6a-c478-4b05-9ffd-a0a05546f811_033
7,— Уже три години.,5,0.410,-2.08,7,0.74,f98bc710-ba1d-43c4-bdc0-8d161470d92e.mp3,83.500-86.200,f98bc710-ba1d-43c4-bdc0-8d161470d92e_003
8,— На нього йому.,5,0.416,-2.67,7,0.76,b8609275-de02-413a-8f45-23144daa202a.mp3,1070.300-1077.200,b8609275-de02-413a-8f45-23144daa202a_045
9,"— Ти не знаєш, що я хочу!",5,0.419,-2.54,12,0.83,c2eb0000-0aff-0242-5378-08d9ea8c52be.mp3,0.000-2.400,c2eb0000-0aff-0242-5378-08d9ea8c52be_000


## Експорт для ручної розмітки

CSV з порожньою колонкою `hallucination`: 1 — галюцинація, 0 — справжнє мовлення.

In [7]:
destination = HERE / f"hallucination_review_{marks_path.parent.name}.csv"
export = sample[COLUMNS].assign(hallucination="")
export.to_csv(destination, index=False)
print(f"{len(export)} рядків -> {destination}")

200 рядків -> /home/romardoss/asr/asr_ssl/pseudo-labelling/confidences/hallucination_review_model-whisper-m-fixed-leaks-10-epochs-epoch05-01.csv
